# Load Scouting Reports from FanGraphs

This notebook loads raw scouting reports from FanGraphs and prepares them for vector indexing.
We'll save this data as a Delta table, which will later be chunked and embedded using Cohere in the next step.


In [0]:
# Environment Variables
CATALOG = "alexander_booth"
SCHEMA = "cohere_demo"
TABLE = "fangraphs_mlb_scouting_reports"
TABLE_CHUNKS = "fangraphs_mlb_scouting_reports_chunked"

In [0]:
# Imports
import requests
import pandas as pd

In [0]:
# Set parameters for the scouting board type and season
pos = "all"  
season = 2025  # The season/year of interest
board_type = "prospect"  # Options: "prospect" (preseason), "updated" (midseason), "mlb" (draft), "int" (international)

# Format the draft parameter based on the season and board type
draft = f"{season}{board_type}"

# Construct the FanGraphs API URL for the selected board
url = f"https://www.fangraphs.com/api/prospects/board/data?draft={draft}&season={season}"
print(url)


https://www.fangraphs.com/api/prospects/board/data?draft=2025prospect&season=2025


In [0]:
# Fetch raw data from the FanGraphs prospect board API
response = requests.get(url)
data = response.json()

# Convert to Pandas DataFrame
df = pd.DataFrame(data)
print(df.shape)
df.head()

(933, 120)


,llevel,mlevel,minorMasterId,playerName,playerNameRoute,UPURL,RPM,Vel,cRisk,cETA,cFV,cWeight,cHeight,cOVR,cORG,cDraftSchool,Hit,Game,Raw,Spd,Fld,Arm,FB,SL,CB,CH,SPL,CT,CMD,cSeason,playerPageSortOrder,ID,FirstName,LastName,Position,Age,Team,Season,Type,PlayerId,...,Levers,FBType,IsVisible,options,servicetime,cCollegeCommit,pHit,fHit,pGame,fGame,pRaw,fRaw,pSpd,fSpd,pFld,fFld,pArm,Draft_Rnd,School_Type,HS_State,College_Commit,Pitch_Sel,Bat_Ctrl,HardHit%,Versatility,Contact_Style,Amateur_Rk,pCB,fCB,pCH,fCH,bRPM,fRPM,Agent,TJDate,pCT,fCT,Trend,Bonus_Class_Rk,Pos_Rk
0,MLB,MLB,sa3025686,Roki Sasaki,Roki Sasaki,/players/roki-sasaki/35323/stats?position=P,,101,High,2025,65,203,"6' 3""",1,1,Japan,,,,,,,45 / 55,55 / 60,,,80 / 80,,40 / 50,2025,0,2424904,Roki,Sasaki,SP,23.55,LAD,2025,2025 Report,35323,...,Long,Downhill,Y,3,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AAA,AAA,sa3020211,Roman Anthony,Roman Anthony,/players/roman-anthony/sa3020211/stats?positio...,,NaN,High,2026,60,205,"6' 2""",2,,Stoneman Douglas HS (FL),40 / 45,55 / 70,70 / 80,55 / 50,50 / 60,NaN,,,,,,,,2025,0,2424480,Roman,Anthony,RF,21.0222222,BOS,2025,2025 Report,sa3020211,...,Long,NaN,Y,Dec'26,,Ole Miss,40.0,45.0,55.0,70.0,70.0,80.0,55.0,50.0,50.0,60.0,60.0,2.0,HS,FL,Ole Miss,55.0,45.0,0.53,OF,InsideOut,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MLB,MLB,sa3022882,Dylan Crews,Dylan Crews,/players/dylan-crews/33541/stats?position=OF,,NaN,Med,2025,60,203,"6' 0""",3,,LSU,45 / 50,50 / 60,60 / 60,70 / 70,40 / 45,NaN,,,,,,,,2025,0,2425549,Dylan,Crews,CF,23.2361111,WSN,2025,2025 Report,33541,...,Med,NaN,Y,3,0.035,NaN,45.0,50.0,50.0,60.0,60.0,60.0,70.0,70.0,40.0,45.0,55.0,1.0,4YR,FL,NaN,50.0,50.0,0.46,OF,InsideOut,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AA,AA,sa3021069,Sebastian Walcott,Sebastian Walcott,/players/sebastian-walcott/sa3021069/stats?pos...,,NaN,High,2027,60,190,"6' 4""",4,,Bahamas,30 / 45,45 / 70,60 / 80,45 / 45,40 / 50,NaN,,,,,,,,2025,0,2425465,Sebastian,Walcott,SS,19.1861111,TEX,2025,2025 Report,sa3021069,...,Long,NaN,Y,Dec'27,,NaN,30.0,45.0,45.0,70.0,60.0,80.0,45.0,45.0,40.0,50.0,60.0,NaN,J2,NaN,NaN,50.0,40.0,0.37,SS/3B,Pull,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AAA,AAA,sa3015716,Samuel Basallo,Samuel Basallo,/players/samuel-basallo/sa3015716/stats?positi...,,NaN,High,2026,60,230,"6' 3""",5,1,Dominican Republic,30 / 40,55 / 70,70 / 80,40 / 40,30 / 45,NaN,,,,,,,,2025,0,2424430,Samuel,Basallo,C,20.7722222,BAL,2025,2025 Report,sa3015716,...,Long,NaN,Y,Dec'25,,NaN,30.0,40.0,55.0,70.0,70.0,80.0,40.0,40.0,30.0,45.0,60.0,NaN,J2,NaN,NaN,30.0,55.0,0.49,C/1B,PoleToPole,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [0]:
# Combine TLDR, Summary, and Ovr_Summary into a unified text block for embedding
def combine_text(row):
    parts = []

    # Include the TLDR section if it exists
    if pd.notnull(row["TLDR"]):
        parts.append("TLDR:\n" + row["TLDR"])

    summary = row["Summary"]
    ovr_summary = row["Ovr_Summary"]

    # Handle cases where Summary is present
    if pd.notnull(summary):
        parts.append("Full Report:\n" + summary)

        # Only add Ovr_Summary if it's present and not identical to Summary
        if pd.notnull(ovr_summary) and summary != ovr_summary:
            parts.append(ovr_summary)

    # If only Ovr_Summary exists (no Summary), include it as the full report
    elif pd.notnull(ovr_summary):
        parts.append("Full Report:\n" + ovr_summary)

    # Join all sections with spacing
    return "\n\n".join(parts)


In [0]:
# Execute the function on each row
df["Combined_Scouting_Report"] = df.apply(combine_text, axis=1)
print(df.Combined_Scouting_Report[0])

TLDR:
Peak Sasaki has talent on par with the best overall prospects in baseball. It seems likely to shine through in a profound way at some point during his MLB tenure, but Sasaki's injury history and 2024 downtick in stuff are notable.

Full Report:
Sasaki hasn't looked quite like peak Roki so far in 2025. He has struggled, sometimes pretty badly, to adjust to the MLB baseball. His fastball has lost vertical movement compared to how it looked in Japan, and Sasaki has had problematic rashes of wildness in his first couple of outings. His slider, however, looks better than it did last season, and his splitter has been every bit the all-world pitch described here and elsewhere for the last couple of years. It's important that at least one of Sasaki's command and/or fastball ride improve in order for this grade to have been appropriate. Better ride will help enable him to miss bats without pinpoint command, while pinpoint command would allow his fastball to play despite a lack of movement

In [0]:
# Convert Pandas DataFrame to Spark DataFrame
sdf = spark.createDataFrame(df)

# Write to Table
sdf.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE}")

In [0]:
# Enable Change Data Feed (CDF) on the scouting reports table using env vars
spark.sql(f"""
  ALTER TABLE `{CATALOG}`.`{SCHEMA}`.`{TABLE}`
  SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")


DataFrame[]

---

✅ **Next Step:** Proceed to the next notebook to chunk these reports and prepare them for embedding with Cohere.
